<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Select_for_4ang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Data Cleaner**

In [ ]:
# @title
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from google.colab import drive
import pandas as pd
import os
from open_dataset_store import OpenDatasetStore
import re
import plotly.io as pio
pio.renderers.default = "colab"

def get_columns(df, devices, imus, axes):
    # Convert single inputs (like the integer 1) into lists so we can loop through them
    if not isinstance(devices, list): devices = [devices]
    if not isinstance(imus, list): imus = [imus]
    if not isinstance(axes, list): axes = [axes]

    valid_columns = []

    for dev in devices:
        for imu in imus:
            for axis in axes:
                # Dynamic Regex Pattern Breakdown:
                # ^.*?    : Matches ANY text at the start (e.g., 'device', 'OrbiX-Node-')
                # (?<!\d) : Ensures the device number isn't immediately preceded by another number
                # {dev}   : Injects your device number (e.g., 1)
                # _IMU    : Matches the literal string '_IMU'
                # {imu}   : Injects your IMU number (e.g., 2)
                # _{axis}$: Matches the underscore, your axis, and the exact end of the string
                pattern = rf"^.*?(?<!\d){dev}_IMU{imu}_{axis}$"

                # Check every column in the dataframe against our regex pattern
                for col in df.columns:
                    if re.match(pattern, col):
                        valid_columns.append(col)

    return valid_columns

def extract_and_rename_imus(df, imu_mapping):
  # The 6 axes we want to extract for every single sensor
  axes = ['AX', 'AY', 'AZ', 'GX', 'GY', 'GZ']

  # Dictionary to store our newly renamed columns before building the DataFrame
  new_data = {}

  # Keep the Timestamp if it exists in your original dataset
  if 'Timestamp' in df.columns:
      new_data['Timestamp'] = df['Timestamp']

  # Loop through the mapping dictionary you provide
  for target_name, config in imu_mapping.items():
      dev = config['device']
      imu = config['imu']

      for axis in axes:
          # Dynamic Regex: Looks for any prefix, followed by your exact device #, _IMU, imu #, and axis.
          # (?<!\d) prevents device '1' from accidentally matching device '11'
          pattern = rf"^.*?(?<!\d){dev}_IMU{imu}_{axis}$"

          # Find the exact column name in your original DataFrame that matches
          matched_cols = [col for col in df.columns if re.match(pattern, col)]

          if matched_cols:
              original_col = matched_cols[0]
              # Create the new name (e.g., 'RKP_AX')
              new_col_name = f"{target_name}_{axis}"
              # Copy the data over
              new_data[new_col_name] = df[original_col]
          else:
              print(f"⚠️ Warning: Missing data for {target_name} ({axis}) -> Looked for Device {dev}, IMU {imu}")

  # Create and return the final dataframe
  return pd.DataFrame(new_data)

drive.mount('/content/drive', force_remount=True)
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"

store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

summary_stats = store.summary()

In [ ]:
store.list_entities(entity_type='subjects')

In [ ]:
store.list_entries(entry_type='experiments')

# Get Data

In [ ]:
entry_id='experiments_0001'

In [ ]:
df = store.get_entry_raw_data(entry_type='experiments', entry_id=entry_id)
sum  = store.get_df_summary(df, detailed=True)

# View Devices

In [ ]:
store.summary_plot(df, columns=get_columns(df, 1, [1,2], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 2, [1,2], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 3, [1,2], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 4, [1,2], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 6, [2,3], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 7, [2,3], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 8, [2,3], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

In [ ]:
store.summary_plot(df, columns=get_columns(df, 10, [2,3], 'AX'),numeric_plots=['violin', 'scatter'],separate_plots=True)

# Extract Columns

In [ ]:
# @title
my_sensor_setup = {
    'RKP': {'device': 1, 'imu': 1},
    'REP': {'device': 2, 'imu': 2},
    'LKP': {'device': 3, 'imu': 1},
    'LEP': {'device': 4, 'imu': 2},
    'RED': {'device': 6, 'imu': 1},
    'LED': {'device': 7, 'imu': 2},
    'LKD': {'device': 8, 'imu': 1},
    'RKD': {'device': 10, 'imu': 2}
}

selected_df = extract_and_rename_imus(df, my_sensor_setup)

display(selected_df.head())

In [ ]:
# @title Save
store.add_processed_data(entry_type='experiments',entry_id=entry_id, df=selected_df, tag='selected_for_4ang',comment='')